# Exploring the occupancy permit data (layer 24)

There is no data dictionary for this layer, so before writing the enrichment script I looked at the data to answer a few questions:

1. What does the layer contain, and how complete is it?
2. How should the 2026 date rule work ("applications or issued permits")?
3. Are the two permit types spelled as in the brief?
4. Which column identifies the utility, and how are the names written?
5. Which column identifies a permit?
6. How many locations does a permit have?

Each section ends with what I decided and why.

In [1]:
import pandas as pd

from step0_rest_client import fetch_layer

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)

LAYER_24 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_APPS/DDOT_TOPS/MapServer/24"

## 1. First look

I download the whole layer with no filter, so I can see everything before deciding how to filter it.

In [2]:
permits = fetch_layer(LAYER_24)
print(f"{len(permits):,} rows, {permits.shape[1]} columns")

20,039 rows, 116 columns


In [3]:
# How full is each column? (share of rows that are not blank)
filled = permits.notna().mean().sort_values(ascending=False).round(3)
print("Columns filled on every row:", (filled == 1).sum())
print("Columns that are completely empty:", (filled == 0).sum())
filled.to_frame("share_filled")

Columns filled on every row: 39
Columns that are completely empty: 8


,share_filled
OBJECTID,1.000
MeterPaidYN,1.000
AddressLockedYN,1.000
WorkLocationAddressID,1.000
WorkLocationFullAddress,1.000
PermitteeName,1.000
WorkLocationWard,1.000
WorkLocationANC,1.000
WorkLocationSMD,1.000
WorkLocationZip,1.000


In [4]:
# The columns that matter for this task
key_columns = [
    "TrackingNumber", "PermitNumber", "EventTypeDescription", "StatusDescription",
    "ApplicationDate", "IssueDate", "ApplicantCompany", "PermitteeName", "OwnerName",
    "WorkLocationFullAddress", "WorkLocationAddressID", "XCoord", "YCoord", "WorkLocationWard",
]
permits[key_columns].head(5)

,TrackingNumber,PermitNumber,EventTypeDescription,StatusDescription,ApplicationDate,IssueDate,ApplicantCompany,PermitteeName,OwnerName,WorkLocationFullAddress,WorkLocationAddressID,XCoord,YCoord,WorkLocationWard
0,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1700 - 1799 BLOCK OF EAST CAPITOL STREET,1433118,401745.69,135773.22,7
1,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1720 - 1799 BLOCK OF C STREET NE,1433119,401784.36,136161.68,7
2,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1800 - 1819 BLOCK OF C STREET NE,1433120,401859.92,136161.83,7
3,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1800 - 1899 BLOCK OF EAST CAPITOL STREET,1433121,401897.81,135773.26,7
4,11145028,PA11145028,Parade,Permit Expired,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00,Eastern Senior High School,Ronald Nelson,Eastern HS (DCPS),1820 - 1899 BLOCK OF C STREET NE,1433122,401935.17,136161.85,7


In [5]:
# Date range of the whole layer
permits[["ApplicationDate", "IssueDate"]].agg(["min", "max", "count"])

,ApplicationDate,IssueDate
min,2025-06-25 12:46:55-04:00,2025-07-08 17:06:54-04:00
max,2026-10-13 04:06:04-04:00,2026-10-13 04:06:30-04:00
count,20039,14173


## 2. Dates

The brief says: *"Permit applications or issued permits should be equal to or after January 1, 2026."*

I want to see how many rows fall into each combination of application date and issue date, and whether the data is spread evenly through 2026.

In [6]:
start = pd.Timestamp("2026-01-01", tz="America/New_York")

applied_2026 = permits["ApplicationDate"] >= start
issued_2026 = permits["IssueDate"] >= start
not_issued = permits["IssueDate"].isna()

groups = pd.Series("applied and issued before 2026", index=permits.index)
groups[applied_2026 & issued_2026] = "applied 2026, issued 2026"
groups[applied_2026 & not_issued] = "applied 2026, not issued yet"
groups[~applied_2026 & issued_2026] = "applied 2025, issued 2026"
groups[~applied_2026 & not_issued] = "applied 2025, not issued yet"

groups.value_counts().to_frame("rows")

,rows
"applied 2026, issued 2026",13779
"applied 2026, not issued yet",5860
applied and issued before 2026,382
"applied 2025, issued 2026",12
"applied 2025, not issued yet",6


In [7]:
# The same split for the two permit types in the brief
two_types = permits["EventTypeDescription"].isin(["Construction Staging Area", "Mobile Crane Work Zone"])
groups[two_types].value_counts().to_frame("rows")

,rows
"applied 2026, issued 2026",3800
"applied 2026, not issued yet",3387
"applied 2025, issued 2026",7
"applied 2025, not issued yet",5


In [8]:
# Applications per week around the start of 2026: is anything missing?
# (counting unique applications, not location rows)
by_application = permits.drop_duplicates("TrackingNumber").set_index("ApplicationDate")
weekly = by_application.resample("W-SUN")["TrackingNumber"].count()
weekly["2025-11-01":"2026-03-15"].to_frame("applications")

,applications
ApplicationDate,
2025-11-02 00:00:00-04:00,0
2025-11-09 00:00:00-05:00,1
2025-11-16 00:00:00-05:00,3
2025-11-23 00:00:00-05:00,2
2025-11-30 00:00:00-05:00,2
2025-12-07 00:00:00-05:00,2
2025-12-14 00:00:00-05:00,1
2025-12-21 00:00:00-05:00,1
2025-12-28 00:00:00-05:00,2


In [9]:
# Applications per month for the whole layer
monthly = by_application.resample("MS")["TrackingNumber"].count()
monthly.index = monthly.index.strftime("%Y-%m")
monthly.to_frame("applications")

,applications
ApplicationDate,
2025-06,1
2025-07,0
2025-08,3
2025-09,8
2025-10,1
2025-11,8
2025-12,7
2026-01,16
2026-02,64


In [10]:
# Earliest applications in 2026
permits.loc[applied_2026, ["TrackingNumber", "ApplicationDate", "EventTypeDescription"]].nsmallest(5, "ApplicationDate")

,TrackingNumber,ApplicationDate,EventTypeDescription
399,11181864,2026-01-13 18:59:26-05:00,Construction Staging Area
400,11181864,2026-01-13 18:59:26-05:00,Construction Staging Area
401,11181864,2026-01-13 18:59:26-05:00,Construction Staging Area
402,11182398,2026-01-16 15:47:45-05:00,Construction Staging Area
403,11182398,2026-01-16 15:47:45-05:00,Construction Staging Area


In [11]:
# Status of the 2026 rows for the two permit types
permits.loc[two_types & (applied_2026 | issued_2026), "StatusDescription"].value_counts().to_frame("rows")

,rows
StatusDescription,
Issued,1991
Permit Expired,1794
Approved (Pending Payment),1398
Not Paid,744
Assigned,706
Resubmit,340
Rejected,120
Withdrawn,75
Suspended,11


In [12]:
# Why so few early-2026 applications? Check the expiration dates.
one_per_application = permits.drop_duplicates("TrackingNumber")
print("Earliest expiration date in the layer:", one_per_application["ExpirationDate"].min())
one_per_application["ExpirationDate"].dt.strftime("%Y-%m").value_counts().sort_index().to_frame("applications")

Earliest expiration date in the layer: 2026-08-30 04:00:00-04:00


,applications
ExpirationDate,
2026-08,402
2026-09,4864
2026-10,4327
2026-11,1766
2026-12,801
2027-01,197
2027-02,91
2027-03,105
2027-04,73


**What section 2 shows**

- **The layer only keeps recent permits.** No permit in it expired before 30 August 2026, about six weeks before the data was pulled (13 October 2026). Older permits seem to be dropped from this nightly extract after they expire. That is why applications climb from 16 in January to over 5,000 in September: early 2026 is not quiet, most of it has already been removed.
- **So "calendar year 2026" can only mean "2026 permits still in the layer".** Permits that expired before about 30 August 2026 are missing, and the result is far from complete for January to August. There is no other occupancy permit layer in the service to fill the gap. This goes in the README as the main limitation, and the CSV records the date the data was pulled.
- **The date rule.** For the two permit types: 3,800 rows were applied for and issued in 2026, 3,387 were applied for in 2026 and are not issued yet, and 7 were applied for in 2025 and issued in 2026. Reading the brief as "applied in 2026 **or** issued in 2026" keeps all three groups (7,194 rows). Only 5 rows (applied in 2025, still not issued) fall outside.
- **Statuses** range from Issued and Expired to Rejected (120 rows), Withdrawn (75) and Revoked (3). The brief does not ask to exclude any status. Decision below.

**Decisions from section 2**

- **Date rule:** keep a location if its application date **or** its issue date is on or after 1 January 2026. This follows the brief's wording and keeps the 7 locations applied for in late 2025 but issued in 2026.
- **Statuses:** keep every status. The brief asks for "permit applications or issued permits", and a rejected or withdrawn application is still an application. The output includes `StatusDescription` so the Associate Director can filter, for example to leave out Rejected, Withdrawn and Revoked.
- **Coverage:** the result covers 2026 permits still in the layer on the day the data was pulled. That date is recorded in the output.

## 3. Permit types

The brief asks for two types: "Construction Staging Area" and "Mobile Crane Work Zone". I check they are spelled exactly like that in the data, and that no similar types (other spellings, "Other" entries) should be included too.

In [13]:
in_2026 = applied_2026 | issued_2026
permits.loc[in_2026, "EventTypeDescription"].value_counts().to_frame("rows")

,rows
EventTypeDescription,
Construction Staging Area,6938
ENP For DDOT Contractors,3107
Moving Truck,2833
Lead Free DC,1713
Banner and Seasonal Display,1113
Other Special Events,814
Other Reserved Parking,528
Roll Off Debris Container,384
After-Hours Activities,344


In [14]:
# Any type that mentions staging or crane, in any spelling?
types = permits["EventTypeDescription"].dropna().unique()
[t for t in types if "staging" in t.lower() or "crane" in t.lower()]

['Construction Staging Area', 'Mobile Crane Work Zone']

In [15]:
# Rows typed "Other" describe themselves in OtherEventType. Do any of them mention staging or cranes?
other = permits.loc[in_2026, "OtherEventType"].dropna()
other[other.str.contains("stag|crane", case=False)].value_counts().to_frame("rows")

,rows
OtherEventType,
Union Stage Events,3
UHG VIP Meeting - SUV/MPD Staging,2
Crew staging area,1
Crane truck parking,1


In [16]:
# Section 1 found types missing from the layer's own code list (EventTypeDescription_label is blank).
# Which types are those, and are our two among them?
unlisted = permits.loc[permits["EventTypeDescription_label"].isna(), "EventTypeDescription"]
unlisted.value_counts().to_frame("rows")

,rows
EventTypeDescription,
Lead Free DC,1713
Banner and Seasonal Display,1113
Small Cell Maintenance 1 Yr,419
After-Hours Activities,344
Block Party,250
Aerial Bridge Maintenance Access Permit,6
WMATA-Bus Bridge,1


**What section 3 shows**

- Both types exist with exactly the brief's spelling: Construction Staging Area (6,938 rows in 2026) and Mobile Crane Work Zone (256). No other type mentions staging or cranes.
- Among rows typed "Other", only 2 describe something close ("Crew staging area", "Crane truck parking"). The other matches are unrelated ("Union Stage Events", MPD staging).
- The types missing from the layer's code list (Lead Free DC, Banner and Seasonal Display, etc.) don't include ours, so the oddity from section 1 doesn't affect this task.
- Worth knowing: utilities also work under other permit types. "Lead Free DC" (1,713 rows) is DC Water's lead pipe replacement programme, and "Manhole Access" sounds like utility work too. The brief limits the request to the two types, so these are left out. I mention them in the README in case the Associate Director wants a wider view.

**Decision:** filter on the two exact type names. The 2 "Other" rows are not added, because the brief names the types explicitly and free-text "Other" entries are not a reliable category.

## 4. Utilities

The brief asks for three utilities: DC Water, PEPCO and Washington Gas. There is no utility column, so the name has to come from one of the company columns. In section 1 the applicant was a permit agent filing for another company, so I check all three name columns.

From here on I only look at rows that pass the date and permit type rules.

In [17]:
in_scope = permits[in_2026 & two_types].copy()
print(f"{len(in_scope):,} location rows, {in_scope['TrackingNumber'].nunique():,} applications")

name_columns = ["ApplicantCompany", "PermitteeName", "OwnerName"]

7,194 location rows, 4,543 applications

In [18]:
# Most common names in each column (one count per application, not per location)
one_per_app = in_scope.drop_duplicates("TrackingNumber")
for col in name_columns:
    print(f"\n=== {col} ===")
    print(one_per_app[col].value_counts().head(15).to_string())


=== ApplicantCompany ===
ApplicantCompany
DCWater/CIP                       1087
Wash Gas & Light Co.               911
N/A                                907
Potomac Electric Power Company     620
DCWater/OPS                        456
Metropolitan Permit Services        77
DFM Development Services, LLC       69
Comcast                             67
Stokes Plans & Permitting           36
LNDC Inc                            31
Department of General Services      29
Pepco                               27
Skanska Flatiron Joint Venture      24
Crown Castle NG Atlantic LLC        23
Permit Me LLC                       21

=== PermitteeName ===
PermitteeName
Fort Myer Construction Corporation                                       747
Potomac Electric Power Company                                           411
DCWater/OPS                                                              301
DONALD HILL                                                              258
WASH GAS & LIGHT CO.       

In [19]:
# Every name that looks like one of the three utilities, in any column
# ("dc ?water" also catches "DCWater/CIP" and "DCWater/OPS", written without a space)
patterns = {
    "DC Water": r"dc ?water|wasa|water and sewer|water & sewer",
    "PEPCO": r"pepco|potomac electric",
    "Washington Gas": r"washington gas|wash gas|wgl",
}
for utility, pattern in patterns.items():
    print(f"\n##### {utility} #####")
    for col in name_columns:
        hits = one_per_app.loc[one_per_app[col].str.contains(pattern, case=False, na=False), col]
        if len(hits):
            print(f"  {col}:")
            print(hits.value_counts().to_string().replace("\n", "\n    ").join(["    ", ""]))


##### DC Water #####
  ApplicantCompany:
    ApplicantCompany
    DCWater/CIP    1087
    DCWater/OPS     456
    DCWater           1
  PermitteeName:
    PermitteeName
    DCWater/OPS    301
    DCWATER/OPS    155
    DCWater/CIP    119
    DC Water        29
  OwnerName:
    OwnerName
    DC Water       775
    DCWATER/OPS    155
    DCWater/OPS    147
    DCWater/CIP     33

##### PEPCO #####
  ApplicantCompany:
    ApplicantCompany
    Potomac Electric Power Company    620
    Pepco                              27
  PermitteeName:
    PermitteeName
    Potomac Electric Power Company                                   411
    POTOMAC ELECTRIC POWER COMPANY                                    46
    Pepco Holdings Inc                                                30
    Pepco                                                             11
    Pepco Holdings Inc.                                                6
    Pepco holdings inc                                                 1
  

In [20]:
# Which utility does each column point to?
def which_utility(name):
    if pd.isna(name):
        return None
    found = [u for u, pattern in patterns.items() if pd.Series([name]).str.contains(pattern, case=False).iloc[0]]
    return found[0] if len(found) == 1 else ("several" if found else None)

for col in name_columns:
    one_per_app[col + "_utility"] = one_per_app[col].map(which_utility)

utility_cols = [c + "_utility" for c in name_columns]
combos = one_per_app[utility_cols].fillna("-").value_counts()
combos.to_frame("applications")

applications
ApplicantCompany_utility PermitteeName_utility OwnerName_utility              
-                        -                     -                          1441
DC Water                 -                     DC Water                    746
Washington Gas           Washington Gas        -                           460
                         -                     -                           444
PEPCO                    PEPCO                 PEPCO                       429
DC Water                 DC Water              DC Water                    364
                                               -                           240
                         -                     -                           194
PEPCO                    -                     -                           127
                         PEPCO                 -                            77
                         -                     PEPCO                        14
Washington Gas           Washington Gas        Washington Gas                7

In [21]:
# Do the columns ever disagree (two different utilities on the same application)?
def utilities_named(row):
    return set(row.dropna()) - {"several"}

named = one_per_app[utility_cols].apply(utilities_named, axis=1)
print("Applications naming more than one utility:", (named.map(len) > 1).sum())
print("Applications naming exactly one utility:  ", (named.map(len) == 1).sum())

Applications naming more than one utility: 0
Applications naming exactly one utility:   3102


In [22]:
# When a utility is the applicant or owner, who is the permittee? (contractors doing the work)
utility_app = one_per_app[named.map(len) == 1]
utility_app.loc[utility_app["PermitteeName_utility"].isna(), "PermitteeName"].value_counts().head(10).to_frame("applications")

,applications
PermitteeName,
Fort Myer Construction Corporation,747
DONALD HILL,258
EDWIN LANE,159
Department of Energy & Environment Department of Energy & Environment,52
Capitol Paving,47
Sagres Construction,25
B FRANK JOY,25
Michael Milbourne,25
Sagres Construction,25


In [23]:
# The reverse: names that are NOT matched in any column, but appear often. Could any be a utility?
not_matched = one_per_app[named.map(len) == 0]
for col in name_columns:
    print(f"\n=== {col} (not matched) ===")
    print(not_matched[col].value_counts().head(8).to_string())


=== ApplicantCompany (not matched) ===
ApplicantCompany
N/A                               907
Metropolitan Permit Services       77
DFM Development Services, LLC      69
Comcast                            67
Stokes Plans & Permitting          36
LNDC Inc                           31
Department of General Services     29
Skanska Flatiron Joint Venture     24

=== PermitteeName (not matched) ===
PermitteeName
Comcast of the District, LLC                                 30
Comcast Of the District, LLC Comcast Of the District, LLC    25
Crown Castle NG Atlantic LLC                                 23
District Properties Inc.                                     23
lamberts cable                                               21
PAUL KILLMER                                                 19
Mathew Butt                                                  18
Ace Utilities Inc.                                           18

=== OwnerName (not matched) ===
OwnerName
Comcast of the District, LLC     

In [24]:
# Result if a utility named in ANY of the three columns counts
one_per_app["utility"] = named.map(lambda s: next(iter(s)) if len(s) == 1 else None)
in_scope = in_scope.merge(one_per_app[["TrackingNumber", "utility"]], on="TrackingNumber", how="left")

summary = pd.DataFrame({
    "applications": one_per_app["utility"].value_counts(),
    "locations": in_scope["utility"].value_counts(),
})
summary.loc["Total"] = summary.sum()
summary

,applications,locations
utility,,
DC Water,1544,1887
Washington Gas,911,1666
PEPCO,647,1373
Total,3102,4926


In [25]:
# How much does checking all three columns add, compared with ApplicantCompany alone?
applicant_only = one_per_app["ApplicantCompany_utility"].notna().sum()
any_column = one_per_app["utility"].notna().sum()
print(f"ApplicantCompany only: {applicant_only:,} applications")
print(f"Any of the three:      {any_column:,} applications (+{any_column - applicant_only})")

ApplicantCompany only: 3,102 applications
Any of the three:      3,102 applications (+0)


**What section 4 shows**

- **The applicant column carries the utility.** Every utility application has the utility as `ApplicantCompany`: "DCWater/CIP" (capital projects) or "DCWater/OPS" (operations), "Wash Gas & Light Co.", and "Potomac Electric Power Company" or "Pepco". The permittee is often a contractor doing the work (Fort Myer Construction, Capitol Paving, Sagres Construction), and the owner is often the utility again.
- **Names are written many ways.** DC Water appears as "DCWater/CIP", "DCWATER/OPS" and "DC Water". PEPCO appears as "Potomac Electric Power Company", "Pepco Holdings Inc." and "PEPCO INC". Washington Gas is almost always "Wash Gas & Light Co.", and only twice "Washington Gas". A search for the brief's exact names would miss most of them. My first pattern missed "DCWater/…" (no space) until I loosened it.
- **No application names two different utilities**, so there is no conflict to resolve.
- **Other utility-like companies are not in scope:** Comcast and Crown Castle (telecoms), and "Ace Utilities Inc." (a contractor).
- **Result:** 3,102 applications and 4,926 locations. DC Water has 1,544 applications (1,887 locations), Washington Gas 911 (1,666) and PEPCO 647 (1,373). 1,441 applications are from other companies and are dropped.

**Decision:** a location is kept if any of the three name columns matches one of these patterns (case-insensitive):

| Utility | Pattern |
|---|---|
| DC Water | `dc ?water`, `wasa`, `water and sewer` |
| PEPCO | `pepco`, `potomac electric` |
| Washington Gas | `washington gas`, `wash gas`, `wgl` |

Today the applicant column alone finds all 3,102 applications. Checking all three costs nothing and protects against future rows where a utility is only named as permittee or owner. The output gets a clean `Utility` column with one of the three names, and the pipeline's QC step reports the count per utility on every run.